# LAB 4 - MD Analysis
**Analysis of an MD simulation with GROMACS and Python**


Authors:
    
- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Analysis

In this section, we will see some basic analysis that can be done on the result of a MD simulation.
We will do the analysis on the data contained in the folder `data/part_4`. We will need a tpr, so let's create it.

Let's create a working directory and copy those data:

In [ ]:
%%bash
mkdir -p 03-analysis
cp data/part_4/* 03-analysis/
cd 03-analysis
gmx grompp -f dummy.mdp -c penetratin.pdb -p penetratin.top -o topol.tpr > /dev/null 2>&1

<div class="alert alert-block alert-warning"><b><center>WARNING</center></b><br>
    As you have noticed, we have created a new file <b>".tpr"</b> in the previous cell. This is due to the fact that the <b>".tpr"</b>  file, unlike most of the other formats produced by gromacs ('xtc', 'gro', 'trr' etc...) is linked to the version of the software with which it was created. In other words, a <b>".tpr"</b>  created with the <b>"gromacs 2020"</b> version can only be read by <b>"gromacs 2020"</b> and not by the version of <b>"gromacs 2021"</b>.
</div>

First, import and install necessary packages necessary packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

def convert_image(f="plot.eps",o="plot.png"):
    '''
    Function that allows you to convert a file from esp format to png format
    '''
    from PIL import Image
    eps_image = Image.open(f)
    eps_image.load(scale=10)
    eps_image.save(o)

## RMSD

The root-mean-square deviation (RMSD) is a measure of the average distance between the atoms (usually C$\alpha$s or the backbone) of superimposed proteins. Typically RMSD is used as a quantitative measure of similarity between two or more protein structures. In the case of Molecular Dynamics simulations, one of the first analysis that is usually done is the observation of RMSD of protein atoms with respect to a target conformation over time:

$$RMSD (t) = \sqrt{\frac{1}{N}\sum^{N}_{i=1}{\left(r_{i}(t)-r_{ref}\right)^2}}$$

where N is the number of atoms that you are considering (e.g., the total number of C$\alpha$s).

If RMSD is computed using the first frame as reference, we can quantify possible conformational changes from the initial structure used in the simulation. Moreover, we can have a first information about simulation convegernce if the RMSD stabilizes.

Consider a 500 ns long simulation and the RMSD with respect to the initial conformation. The RMSD stabilized around 1 nm starting from 100 ns. Can you say that the conformations at 450 ns and 200 ns are for sure similar?

The answer is no!
That information suggests you that the conformation are similar, but the actual information is that they are "equally distant" from the initial configuration of the system (reference frame). Therefore, it can happen that:

$$ RMSD(200ns, 450ns) >> RMSD(0ns, 450ns) = RMSD(0ns, 200ns)$$

We can test deeper the convergence of the simulation by using another frame as reference, such as the frame at 450ns. If the RMSD is stable and "low" starting from 100 ns (i.e., much lower than the one with respect to the initial frame), then we can be more confident about simulation convergence.

**Note**

The RMSD can be computed to other, non-protein molecules, such as small organic molecules, i.e., ligands. For instance, if we fit the trajectory on the protein and measure the RMSD of the ligand, we obtain information about the ligand position within the protein binding cleft.

In [ ]:
!gmx rms -h

In [ ]:
%%bash
cd 03-analysis/
echo -e "Backbone\nBackbone" | gmx rms -s penetratin.pdb -f penetratin.xtc -o rmsd.xvg -tu ns
echo -e "System" | gmx trjconv -s topol.tpr -f penetratin.xtc -dump 9 -o struct9ns.pdb -tu ns
echo -e "Backbone\nBackbone" | gmx rms -s struct9ns.pdb -f penetratin.xtc -o rmsd_9ns.xvg -tu ns

In [ ]:
# Load your data
time, rmsd = np.loadtxt('03-analysis/rmsd.xvg',comments=['@','#'],unpack=True)
_, rmsd9 = np.loadtxt('03-analysis/rmsd_9ns.xvg',comments=['@','#'],unpack=True)

# Prepare the figure
fig = plt.figure()
ax = fig.add_subplot(111,)

# Plot your data
ax.plot(time,rmsd,label='Ref: 0 ns',c='k')
ax.plot(time,rmsd9,label='Ref: 9 ns',c='r')

# Adjust the plot and personalize
ax.set_xlim(time[0],time[-1])
m = np.max(np.concatenate((rmsd,rmsd9)))*1.05
ax.set_ylim(0,m)
ax.set_xlabel(r"Time [ns]")
ax.set_ylabel(r"RMSD [nm]")
ax.legend(frameon=False)

# This allows you to adjust the figure limits to make your figure nice
# (Recommended but not compulsory)
fig.tight_layout()

# Now, save the plot (if you want)
fig.savefig('03-analysis/rmsd_comparison.jpg',dpi=300,facecolor='white')

## RMSF

The root-mean-square fluctuation (RMSF) from the average is a measure that allows you to quantify the fluctuation of an atom (or a group of atoms) during time.

In the case of Molecular Dynamics simulations, the RMSF of protein residues is computed to identify the most stable and the most flexible parts of the protein structures. Usually, non-structured regions are the most flexible. The RMSF is usually computed as an average for each residue or is computed considering the C$\alpha$s as representative of the whole residues.

The RMSF for the i-th atom is defined as:

$$RMSF(i) = \sqrt{\frac{1}{N}\sum^{N}_{t=1}{\left(r_{i}(t)-r_{mean}\right)^2}}$$

where N is the total number of frames in the trajectory.

In [ ]:
!gmx rmsf -h

In [ ]:
%%bash
cd 03-analysis/
echo "C-alpha" | gmx rmsf -f penetratin.xtc -s topol.tpr -o rmsf.xvg -res

In [ ]:
!cat 03-analysis/rmsf.xvg

In [ ]:
# Load your data
res, rmsf = np.loadtxt('03-analysis/rmsf.xvg',comments=['@','#'],unpack=True)

# Prepare the figure
fig = plt.figure()
ax = fig.add_subplot(111,)

# Plot your data
ax.plot(res,rmsf,c='k')

# Adjust the plot and personalize
ax.set_xlim(res[0],res[-1])
m = np.max(rmsf)*1.05
ax.set_ylim(0,m)
ax.set_xlabel(r"Residue number")
ax.set_ylabel(r"RMSF [nm]")

# This allows you to adjust the figure limits to make your figure nice
# (Recommended but not compulsory)
fig.tight_layout()

# Now, save the plot (if you want)
fig.savefig('03-analysis/rmsf.png',dpi=300,facecolor='white')

**Note**
You can use the rmsf command of GROMACS to generate an average configuration of the protein in a certain portion of the trajectory. This configuration can be used as reference in the RMSD analysis.

Suppose that you take the average configuration in the trajectory from 400ns to 450ns: if the RMSD is "low" and stable also berfore (and after) that interval, it means that during the equilibrium trajectory the conformations are equally distant from the average of time \[400ns; 450ns\]

In [ ]:
%%bash
cd 03-analysis/
echo "Protein" | gmx rmsf -f penetratin.xtc -s topol.tpr -o rmsf_eq.xvg -ox average.pdb -b 8000 -e 9000
echo -e "Backbone\nBackbone" | gmx rms -s average.pdb -f penetratin.xtc -o rmsd_av.xvg -tu ns

In [ ]:
time, rmsd = np.loadtxt('03-analysis/rmsd_av.xvg',comments=['@','#'],unpack=True)
_, rmsd50 = np.loadtxt('03-analysis/rmsd_9ns.xvg',comments=['@','#'],unpack=True)
_, rmsd0 = np.loadtxt('03-analysis/rmsd.xvg',comments=['@','#'],unpack=True)
fig = plt.figure()
ax = fig.add_subplot(111,)
ax.plot(time,rmsd0,c='k',label='Ref: 0 ns')
ax.plot(time,rmsd50,c='r',label='Ref: 9 ns')
ax.plot(time,rmsd,c='b',label='Ref: [8-9]ns')
ax.set_xlim(time[0],time[-1])
m = np.max(np.concatenate((rmsd,rmsd50,rmsd0)))*1.05
ax.set_ylim(0,m)
ax.set_xlabel(r"Time [ns]")
ax.set_ylabel(r"RMSD [nm]")
ax.legend()
fig.tight_layout()
fig.savefig('03-analysis/rmsd_average.png',dpi=300,facecolor='white')

## Radius of Gyration

Radius of gyration or gyradius of a body about the axis of rotation is defined as the radial distance to a point which would have a moment of inertia the same as the body's actual distribution of mass, if the total mass of the body were concentrated there.
In polymer physics, the radius of gyration is used to describe the dimensions of a polymer chain. The radius of gyration of a particular molecule at a given time is defined as:

$$ R_{\mathrm {g} }\ =\ \sqrt{{\frac {1}{N}}\sum _{i=1}^{N}\left(\mathbf {r} _{i}-\mathbf {r} _{\mathrm {COM} }\right)^{2}}$$

In [ ]:
!gmx gyrate -h

In [ ]:
%%bash
cd 03-analysis/
echo "Protein" | gmx gyrate -f penetratin.xtc -s topol.tpr -o rog.xvg

Let's make thing more difficult. Plot both the time distribution of the radius of gyration and an histogram distribution

In [ ]:
time, rog,_,_,_ = np.loadtxt('03-analysis/rog.xvg',comments=['@','#'],unpack=True)
time /= 1000 # gmx gyrate gives the time in ps 
fig = plt.figure(figsize=(6,3))
# in time
ax1 = fig.add_subplot(121,)
ax1.plot(time,rog,c='k')
ax1.set_xlim(time[0],time[-1])
M = np.max(rog)*1.05
m = np.min(rog)*0.95
ax1.set_ylim(m,M)
ax1.set_xlabel(r"Time [ns]")
ax1.set_ylabel(r"$R_g$ [nm]")

# histogram
ax2 = fig.add_subplot(122,)
ax2.hist(rog,color='k',density=True,alpha=0.5)
ax2.set_ylabel(r"Probability density")
ax2.set_xlabel(r"$R_g$ [nm]")
fig.tight_layout()
fig.savefig('03-analysis/radius_of_gyration.png',dpi=300,facecolor='white')

## Ramachandran plot

The allowed combinations of torsional angles ψ and φ for a couple of residues are illustrated in the Ramachandran plot. It serves to represent allowed and disallowed regions of the two torsional angles of each peptide bond in a polypeptide chain.

To compute the torsional angles in GROMACS we need:
1. a tpr file containing the information about all the atoms in the system
2. the coordinates of the system. It can be a single structure or an ensemble (a pdb with multiple frames or an xtc)

In [ ]:
!gmx rama -f 03-analysis/penetratin.xtc -s 03-analysis/topol.tpr -o 03-analysis/rama.xvg

Now, the formatting of the output file is:

$\phi$ $\psi$ res1(time 1)

$\phi$ $\psi$ res2(time 1)

$\phi$ $\psi$ res3(time 1)

$\phi$ $\psi$ res4(time 1)

$\phi$ $\psi$ res1(time 2)

$\phi$ $\psi$ res2(time 2)

...

$\phi$ $\psi$ res4(time N-1)

$\phi$ $\psi$ res1(time N)

$\phi$ $\psi$ res2(time N)

$\phi$ $\psi$ res3(time N)

$\phi$ $\psi$ res4(time N)

**No need to worry:**
Here we have an example of analysis, so you can extract pieces of cose to use in your analysis. Use them wisely!

In [ ]:
# Load data
# You have to load separately floating point numbers and string
phi, psi = np.loadtxt('03-analysis/rama.xvg',comments=['@','#'],usecols=(0,1),unpack=True)
residues = np.loadtxt('03-analysis/rama.xvg',comments=['@','#'],usecols=(2),dtype=str)

# This line searches all the points in the array where the first residue of the protein is present
# Therefore, it contains the information to separate information of different frames
delimiters = np.squeeze(np.argwhere(residues == residues[0]))

In [ ]:
from matplotlib.lines import Line2D # For the legend
from MDAnalysis.analysis.data.filenames import Rama_ref
import time

The data in `Rama_ref` contains information about dihedral $\phi$ and $\psi$ angles on a set of 500 PDB structures taken from [Lovell2003](https://doi.org/https://doi.org/10.1002/prot.10286). This is a numpy array that can be used to create *Allowed* and *Generously Allowed* (which means that contain 90% and 99% of the data points) in the Ramachandran plot.

In [ ]:
from IPython import display
frames = [0,800]
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111,)
# Generate points in the plane to plot the Ramachandran regions
# THESE X,Y POINTS ARE CREATED IN ORDER TO BE COHERENT WITH DATA IN Rama_ref
# DO NOT MODIFY
X, Y = np.meshgrid(np.arange(-180, 180, 4), np.arange(-180, 180, 4))
Z = np.load(Rama_ref)
# THE NUMBERS ON LEVELS ARE DEFINED TO PLOT THE ALLOWED AND GENEROUSLY ALLOWED REGIONS
# DO NOT MODIFY
c = ax.contourf(X, Y, Z, levels=[1, 17, 15000],colors = ['#A1D4FF', '#35A1FF'])

# Set the limits of torsional angles
ax.set_xlim(-180,180)
ax.set_ylim(-180,180)
ax.set_xlabel(r"$\phi$ [°]")
ax.set_ylabel(r"$\psi$ [°]")

#Personalize it with a legend
custom_lines = [Line2D([0], [0], color='#A1D4FF', lw=8),
                Line2D([0], [0], color='#35A1FF', lw=8)]
ax.legend(custom_lines, ['99%', '90%'],loc='lower center',frameon=False)
#if you want to personalize the colors create a list of colors 
colors=['tomato','k']

#return a list of random exadecimal colors based on numpy library in one line
colors=['#'+"".join([np.random.choice(list('0123456789ABCDEF')) for j in range(6)]) for i in range(len(frames))]

plt.ion()
for col,frame in zip(colors,frames):
    ax.scatter(phi[delimiters[frame]:delimiters[frame+1]],psi[delimiters[frame]:delimiters[frame+1]],
              marker='o',s=20,c=col)
    display.display(plt.gcf())
    display.clear_output(wait=True)
    time.sleep(1)
plt.show()


Do you remember what secondary structure is usually present in the region populated by in the above Ramachandran plot?

Is this coherent with our peptide?

## Distance between the extrema

In some molecular systems, the distance between two residues (or group of residues) is an important descriptor of the protein conformation.

Let's assume that there is loop of a protein that, with its movement, opens or closes a binding site. We might define the distance of this loop from the binding site and describe the site opening or closure in terms of such distance.

This was only a very basic example, but it stresses the fact that, in some cases, important information of the molecular systems can be described with simple metrics. Usually, in real life problems, the heard part is finding these metrics from the observation of the system trajectory (do you remember the lessons about metadynamics and collective variables?).

To make an example, let's compute the distance between the two extrema of our peptide. It can be done with GROMACS using the `gmx distance` command. To do so:
1. create an index containing the two groups from which you want to compute the distance
2. use the gromacs function with the option `-select 'com of group "[name group 1]" plus com of group "[name group 2]"`

**Note** that `com` stands for _center of mass_, which means you are computing the distances between the centres of mass of the two groups.

In [ ]:
gmx distance -h

In [ ]:
%%bash
cd 03-analysis
echo "
keep 0
r 43
name 1 E1
r 56
name 2 E2
q" | gmx make_ndx -f penetratin.pdb -o index.ndx
gmx distance -f penetratin.xtc -s topol.tpr -select 'com of group "E1" plus com of group "E2"' \
-n index.ndx -oav distance.xvg

In [ ]:
time, d = np.loadtxt('03-analysis/distance.xvg',comments=['@','#'],unpack=True)
time = time/1000 # gmx distance gives output only in ps
fig = plt.figure(figsize=(6,3))
# in time
ax1 = fig.add_subplot(121,)
ax1.plot(time,d,c='k')
ax1.set_xlim(time[0],time[-1])
M = np.max(d)*1.05
m = np.min(d)*0.95
ax1.set_ylim(m,M)
ax1.set_xlabel(r"Time [ns]")
ax1.set_ylabel(r"$d_{ee}$ [nm]")

# histogram
ax2 = fig.add_subplot(122,)
ax2.hist(d,color='k',density=True,alpha=0.5)
ax2.set_ylabel(r"Probability density")
ax2.set_xlabel(r"$d_{ee}$ [nm]")
fig.tight_layout()
fig.savefig('03-analysis/distance.png',dpi=300,facecolor='white')

## Hydrogen Bonds

Hydrogen bond is a type of dipole-dipole attraction between molecules. It results from the attractive force between a hydrogen atom convalently bonded to a very elecctronegative atom such as N, O or F atom and another very elecctronegative atom.

The Hydrogen Bond analysis aims at identify the number and/or the duration of hydrogen bonds in a system of interest. 

Hydrogen bonds are determined by `GROMACS` function **`gmx hbond`** based on two geometric features:
* the angle Hydrogen - Donor - Acceptor
* the distance Donor - Acceptor (or Hydrogen - Acceptor).

In [ ]:
!gmx hbond -h

Usully one can be interested in exploring the number of hydrogen bonds between the Protein and the solvent (Water) and the Protein with it-self 

In [ ]:
%%bash
cd 03-analysis
echo -e "Protein \n Water \n " | gmx hbond -f penetratin.xtc -s topol.tpr -num hb_water_protein.xvg 
echo -e "Protein \n Protein \n " | gmx hbond -f penetratin.xtc -s topol.tpr -num hb_protein_protein.xvg 

In [ ]:
time, hb_protein_water = np.loadtxt('03-analysis/hb_water_protein.xvg',comments=['@','#'],unpack=True)
_, hb_protein_protein = np.loadtxt('03-analysis/hb_protein_protein.xvg',comments=['@','#'],unpack=True)
time /= 1000

fig = plt.figure(figsize=(11,4))
ax1 = fig.add_subplot(131,)
ax2 = fig.add_subplot(132,sharex = ax1)
ax3 = fig.add_subplot(133,)
ax1.plot(time,hb_protein_water,c='b')
ax2.plot(time,hb_protein_protein,c='r')
ax3.hist(hb_protein_water,color='b',density=True)
ax3.hist(hb_protein_protein,color='r',density=True)
ax1.set_xlim(time[0],time[-1])
ax1.set_xlabel(r"Time [ns]")
ax2.set_xlabel(r"Time [ns]")
ax3.set_xlabel(r"# HB")
ax1.set_ylabel(r"# HB")
ax2.set_ylabel(r"# HB")
ax3.set_ylabel("Probability density")
ax1.set_title('Water-Protein')
ax2.set_title('Protein-Protein')
m = np.min(np.concatenate((hb_protein_water,hb_protein_protein)))*0.95
M = np.max(np.concatenate((hb_protein_water,hb_protein_protein)))*1.05
ax3.set_xlim(m,M)
fig.tight_layout()
fig.savefig('03-analysis/hydrogen_bond.png',dpi=300,facecolor='white')

## Secondary structures Assesment

Protein secondary structure is the three dimensional form of local segments of proteins and refers to the pattern of hydrogen bonds between the carboxyloxygen atoms and amino hydrogen in the peptide backbone.

<img src="imgs/SS.png" width="400" align="center">


Different methodologies and program has been developed to assess the secondary structure from the 3D protein coordinates. 

**`Gromacs`** employ the software **`DSSP`** integrated in the function **`gmx do_dssp`**.

The **`DSSP`** program works by calculating the most likely secondary structure assignment given the 3D structure of a protein. 

It does this by reading the position of the atoms in a protein followed by calculation of the H-bond energy between all atoms. 

The algorithm will discard any hydrogens present in the input structure and calculates the optimal hydrogen positions by placing them at 1.000 Å from the backbone N in the opposite direction from the backbone C=O bond. 

The best two H-bonds for each atom are then used to determine the most likely class of secondary structure for each residue in the protein.

**`gmx do_dssp`** requires two necessary input:
1. **-f** the coordinates of the system. It can be a single structure or an ensemble (.xtc, .trr, .pdb)
1. **-s** the `.tpr` file containing the information about all the atoms in the system

In [ ]:
!gmx dssp -h

In [ ]:
%%bash
cd 03-analysis/
echo "Protein" | gmx dssp -s penetratin.pdb -f penetratin.xtc -tu ns -dt 0.1

Now, let's plot the secondary structure of the peptide along the simulation!

The ```gmx dssp``` command will produce a file called "dssp.dat" that contains the secondary structure of the protein along the simulation. 

***Every line of the file is a frame of the trajectory and contains the secondary structure of each residue of the protein.***

gmx dssp allows using the DSSP algorithm (namely, by detecting specific patterns of hydrogen bonds between amino acid residues) to determine the secondary structure of a protein.

One-symbol secondary structure designations that are used in the output file:

    H — alpha-helix;
    B — residue in isolated beta-bridge;
    E — extended strand that participates in beta-ladder;
    G — 3_10-helix;
    I — pi-helix;
    P — kappa-helix (poly-proline II helix);
    S — bend;
    T — hydrogen-bonded turn;
    = — break;
    ~ — loop (no special secondary structure designation).

In [ ]:
# now let's plot the secondary structure every frame using a different color for each secondary structure

# Load the data
data = np.loadtxt('03-analysis/dssp.dat',comments=['@','#'],dtype=str)

# assign a number to each secondary structure
ss = {'H':0,'B':1,'E':2,'G':3,'I':4,'T':5,'S':6,'P':7,'=':8,'~':9}
ss_translation = {0:'Helix',1:'Beta-Sheet',2:'Strand',3:'3-10 helix',4:'Pi helix',5:'Hydrogen Bond Turn',6:'Bend',7:'Kappa-Helix',8:'Break',9:'Loop'}

# i want to replace all the H letters with 0, all the B letters with 1, etc.
data = [[ss[j] for j in i] for i in data]

In [ ]:
# plot the data as a heatmap
plt.figure(figsize=(10,5))

# Define a set of discrete colors
colors = [
    "#4E79A7",  # Steel Blue
    "#F28E2B",  # Mandarin Orange
    "#E15759",  # Coral Red
    "#76B7B2",  # Seafoam Green
    "#59A14F",  # Fresh Green
    "#EDC948",  # Golden Yellow
    "#AF7AA1",  # Soft Lavender
    "#FF9DA7",  # Pink Salmon
    "#9C755F",  # Mocha Brown
    "#BAB0AC"   # Warm Gray
]

# Create a colormap using the colors
import matplotlib.colors as mcolors
cmap = mcolors.ListedColormap(colors, name='discrete')

plt.imshow(np.array(data).T, cmap=cmap, aspect='auto')
plt.xlabel("Frame")
plt.ylabel("Residue")
plt.title("Secondary structure")
# create a legend outside the plot using the ss keys as labels
plt.legend(handles=[plt.Line2D([0], [0], color=c, lw=4) for c in colors], labels=ss_translation.values(), loc='center left', bbox_to_anchor=(1, 0.5))

plt.savefig('03-analysis/secondary_structure.png',dpi=300,facecolor='white')
plt.tight_layout()
